
**خطة البحث**
 1. شرح الميزة والبيانات 
 2. تحليل البيانات الأولية
 3. تحليل البيانات البصرية الأولية
 4. الرؤى والتبعيات الموجودة
 5. اختيار المقاييس
 6. اختيار النموذج
 7. المعالجة المسبقة للبيانات
 8. التحقق من صحة وتعديل المعلمات الفائقة للنموذج
 9. إنشاء ميزات جديدة ووصف هذه العملية
 10. رسم منحنيات التدريب والتحقق من الصحة
 11. التنبؤ بالعينات الاختبارية أو المحتجزة 
 12. الاستنتاجات 



### الجزء الأول. شرح الميزات والبيانات 
يتم توفير بيانات الإيجار بالساعة الممتدة لمدة عامين. بالنسبة لهذه [المنافسة](https://www.kaggle.com/c/comp180bikeshare)، تتكون [مجموعة التدريب](https://www.kaggle.com/c/comp180bikeshare/data) من أول 16 يومًا من كل شهر، في حين أن مجموعة الاختبار هي اليوم 17-19 من الشهر. يجب عليك التنبؤ بالعدد الإجمالي للدراجات المستأجرة خلال كل ساعة تغطيها مجموعة الاختبار، وذلك باستخدام المعلومات المتاحة قبل فترة الإيجار فقط. أي توقع "العدد" دون استخدام "العدد" أو مكوناته "غير الرسمية" و"المسجلة".
**حقول البيانات*** *التاريخ والوقت* - التاريخ بالساعة + الطابع الزمني
* *الموسم* - 1 = الربيع، 2 = الصيف، 3 = الخريف، 4 = الشتاء
* *عطلة* - ما إذا كان اليوم يعتبر عطلة أم لا
* *يوم عمل* - سواء كان اليوم ليس عطلة نهاية أسبوع أو عطلة
* *الطقس* - 
    1. صافي، قليل السحب، غائم جزئيًا، غائم جزئيًا
    2. ضباب + غائم، ضباب + سحب مكسورة، ضباب + قليل من السحب، ضباب
    3. ثلوج خفيفة، أمطار خفيفة + عاصفة رعدية + سحب متفرقة، أمطار خفيفة + سحب متفرقة
    4. أمطار غزيرة + منصات جليدية + عاصفة رعدية + ضباب، ثلج + ضباب
* *درجة الحرارة* - درجة الحرارة بالدرجة المئوية
* *atemp* - درجة الحرارة "تبدو وكأنها" بالدرجة المئوية
* *الرطوبة* - الرطوبة النسبية
* *سرعة الرياح* - سرعة الرياح
* *غير رسمي* - عدد عمليات الإيجار التي بدأها المستخدمون غير المسجلين
* *المسجل* - عدد عمليات تأجير المستخدمين المسجلين التي بدأت
* *count* - إجمالي عدد الإيجارات



### الجزء الثاني. تحليل البيانات الأولية


In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
import gc

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from catboost import CatBoostRegressor, Pool, cv
from sklearn.metrics import mean_squared_error

import seaborn as sns
from matplotlib import pyplot as plt
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

# Fixing random seed
np.random.seed(17)

print(os.listdir("../input"))

In [ ]:
# Fixing random seed
np.random.seed(17)
# Read data
data_df = pd.read_csv('../input/train_luc.csv')

# Convert to datetime
data_df['datetime'] = pd.to_datetime(data_df['datetime'])

# Sort by datetime
data_df.sort_values(by='datetime')

# Look at the first rows of the training set
data_df.head()

In [ ]:
data_df.shape


*يحتوي القطار على 3 أعمدة مستهدفة: ** "غير رسمي"، "مسجل"، "عدد".**
عمود "العدد" هو مجموع العمودين "غير رسمي" و"المسجل". التحقق من ذلك:*


In [ ]:
(data_df['casual'] + data_df['registered'] - data_df['count']).value_counts()

In [ ]:
# Get info by train
data_df.info()


ممتاز! ليس لدينا نان.


In [ ]:
# Get statistics by train_df
data_df.describe()


### الجزء 3. تحليل البيانات المرئية الأولية



لقد قمت بتقسيم البيانات إلى عينات تدريب وعينات انتظار 


In [ ]:
train_df, test_df, y_train, y_test = train_test_split(data_df.drop(['casual', 'registered', 'count'], axis=1), data_df[['casual', 'registered', 'count']], 
                                                      test_size=0.3, random_state=17, shuffle=True)

In [ ]:
def draw_train_test_distribution(column):
    _, axes = plt.subplots(nrows=1, ncols=2, figsize=(10,3))
    sns.distplot(train_df[column], ax = axes[0], label='train')
    sns.distplot(test_df[column], ax = axes[1], label='test');

In [ ]:
# The distribution of the indicative features
draw_train_test_distribution('season')
draw_train_test_distribution('holiday')
draw_train_test_distribution('workingday')
draw_train_test_distribution('weather');



يتزامن توزيع الميزات الإرشادية ("الموسم" و"العطلة" و"يوم العمل" و"الطقس") في القطار والاختبار. يوجد "طقس" بقيمة 4 في الاختبار وهو غائب في القطار


In [ ]:
test_df[test_df['weather'] == 4]['weather'].count()


يحتوي مثال واحد فقط على "الطقس"=4.  هذا يمكن إهماله


In [ ]:
# The distribution of the numerical features on the train and test
draw_train_test_distribution('temp')
draw_train_test_distribution('atemp')
draw_train_test_distribution('humidity')
draw_train_test_distribution('windspeed')



يتزامن توزيع ميزات المقياس ('درجة الحرارة'، 'درجة الحرارة'، 'الرطوبة'، 'سرعة الرياح') على القطار والاختبار


In [ ]:
def transformation(columnName, func = np.log1p):
    temp_train = pd.DataFrame(index=train_df.index)
    temp_train[columnName] = train_df[columnName].apply(func)

    temp_test = pd.DataFrame(index=test_df.index)
    temp_test[columnName] = test_df[columnName].apply(func)
    
    _, axes = plt.subplots(nrows=1, ncols=2, figsize=(10,3))
    sns.distplot(temp_train, ax = axes[0])
    sns.distplot(temp_test, ax = axes[1]);

In [ ]:
transformation('temp')
transformation('atemp')
transformation('humidity')


بعد تحول التوزيع على Train_df وعلى test_df بدأوا يتشابهون أكثر


In [ ]:
train_df['temp_tr'] = train_df['temp'].apply(np.log1p)
test_df['temp_tr'] = test_df['temp'].apply(np.log1p)
train_df['atemp_tr'] = train_df['atemp'].apply(np.log1p)
test_df['atemp_tr'] = test_df['atemp'].apply(np.log1p)
train_df['humidity_tr'] = train_df['humidity'].apply(np.log1p)
test_df['humidity_tr'] = test_df['humidity'].apply(np.log1p)


### الجزء 4. الرؤى والتبعيات التي تم العثور عليها

In [ ]:
corr = train_df.join(y_train).corr('spearman')
plt.figure(figsize = ( 12 , 10 ))
sns.heatmap(corr,annot=True,fmt='.2f',cmap="YlGnBu");


    يمكننا أن نرى ارتباطًا قويًا بين الأعمدة المستهدفة (ما كان متوقعًا). بين العملاء، يكون الارتباط مع المستخدمين المسجلين أعلى منه مع المستخدمين غير المسجلين. في هذه الحالة، غير مسجل يبحث أكثر في الطقس.
    الارتباطات temp/temp_tr، atemp/atemp_tr، الرطوبة/humidity_tr متساوية. سأستخدم ميزات *_tr، لأن توزيعاتها في التدريب/الاختبار متشابهة.
    **الفكرة: قم ببناء مجموعة باستخدام 3 نماذج ذات أهداف مختلفة!**
    "العطلة" لها علاقة منخفضة بالأهداف. لن أستخدمه.
    "يوم العمل" له ارتباط أقل مع "مسجل" و"عدد" مقارنة بـ "غير رسمي"
    تؤثر "سرعة الرياح" على كل من المستخدمين المسجلين وغير المسجلين
    تأثير "درجة الحرارة" و"atemp" هو نفسه.
    



### الجزء الخامس. اختيار المقاييس
وفقًا لشروط [المنافسة](https://www.kaggle.com/c/comp180bikeshare#evaluation) سأستخدم جذر متوسط الخطأ التربيعي (RMSE)



### الجزء السادس. اختيار النموذج



توصي [الدورة التدريبية](https://www.coursera.org/learn/competitive-data-science) باستخدام نموذج [تعزيز التدرج](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingRegressor.html) باعتباره واحدًا من أقوى النماذج. أريد استخدام مكتبة Catboost، لأنني أريد أن أفهمها. من مكتبة catboost سأستخدم CatBoostRegression، لأن المهمة الحالية مرتبطة بالانحدار.



### الجزء السابع. المعالجة المسبقة للبيانات



* لقد قمت بتغيير ميزات "درجة الحرارة"، و"درجة الحرارة"، و"الرطوبة" في الجزء الثالث. 
* NaN غائب. 
* لن أستخدم OHE لأن CatBoostRegressor يأخذ قائمة من الميزات الفئوية كمعلمة
* سيكون قياس البيانات في CatBoost تلقائيًا


In [ ]:
X_train = train_df.drop(['holiday', 'datetime', 'temp', 'atemp', 'humidity'], axis=1)
X_test = test_df.drop(['holiday', 'datetime', 'temp', 'atemp', 'humidity'], axis=1)


### الجزء 8. التحقق من صحة وتعديل المعلمات الفائقة للنموذج


In [ ]:
cat_features = [0, 1, 2]

X_train_cbr, X_test_cbr, y_train_cbr, y_test_cbr = train_test_split(X_train, y_train, test_size=0.3, random_state=17, shuffle=True)

In [ ]:
from hyperopt import hp, fmin, tpe, STATUS_OK, Trials
import colorama

# the number of random sets of hyperparameters
N_HYPEROPT_PROBES = 100

# hyperparameter sampling algorithm
HYPEROPT_ALGO = tpe.suggest

space ={
        'depth': hp.quniform("depth", 4, 10, 1),
        'learning_rate': hp.loguniform('learning_rate', -3.0, -0.7),
        'l2_leaf_reg': hp.uniform('l2_leaf_reg', 1, 10),
       }

def get_catboost_params(space):
    params = dict()
    params['learning_rate'] = space['learning_rate']
    params['depth'] = int(space['depth'])
    params['l2_leaf_reg'] = space['l2_leaf_reg']
    return params

def objective(space, target_column='count'):
    global obj_call_count, cur_best_rmse

    obj_call_count += 1

    print('\nCatBoost objective call #{} cur_best_acc={:7.5f}'.format(obj_call_count, cur_best_rmse) )

    params = get_catboost_params(space)

    sorted_params = sorted(space.items(), key=lambda z: z[0])
    params_str = str.join(' ', ['{}={}'.format(k, v) for k, v in sorted_params])
    print('Params: {}'.format(params_str) )

    model = CatBoostRegressor(iterations=2000,
                              cat_features = cat_features,
                            learning_rate=params['learning_rate'],
                            depth=int(params['depth']),
                            use_best_model=True,
                            eval_metric='RMSE',
                            l2_leaf_reg=params['l2_leaf_reg'],
                            early_stopping_rounds=50,
                            random_seed=17,
                            verbose=False
                            )
    model.fit(X_train_cbr, y_train_cbr[target_column], 
              eval_set=(X_test_cbr, y_test_cbr[target_column]), 
              verbose=False)
    nb_trees = model.get_best_iteration()

    print('nb_trees={}'.format(nb_trees))

    y_pred = model.predict(X_test_cbr)

    rmse = np.sqrt(mean_squared_error(y_test_cbr[target_column], y_pred))

    print('rmse={}, Params:{}, nb_trees={}\n'.format(rmse, params_str, nb_trees ))

    if rmse<cur_best_rmse:
        cur_best_rmse = rmse
        print(colorama.Fore.GREEN + 'NEW BEST RMSE={}'.format(cur_best_rmse) + colorama.Fore.RESET)


    return{'loss':rmse, 'status': STATUS_OK }

In [ ]:
%%time
obj_call_count = 0
cur_best_rmse = np.inf

trials = Trials()
best = fmin(fn=objective,
                     space=space,
                     algo=HYPEROPT_ALGO,
                     max_evals=N_HYPEROPT_PROBES,
                     trials=trials,
                     verbose=1)

In [ ]:
print('The best params:')
print( best )

In [ ]:
cbr = CatBoostRegressor(random_seed=17, 
                        eval_metric='RMSE', 
                        iterations=2000, 
                        max_depth=best['depth'], 
                        early_stopping_rounds=50, 
                        learning_rate=best['learning_rate'],
                        l2_leaf_reg=best['l2_leaf_reg'],
                       use_best_model=True)
cbr.fit(X_train_cbr, y_train_cbr['count'], 
       eval_set=(X_test_cbr, y_test_cbr['count']), 
       cat_features=cat_features,
       silent=True,
       plot=True);

In [ ]:
rmse_learn = cbr.evals_result_['learn']['RMSE']
rmse_test = cbr.evals_result_['validation_0']['RMSE']

plt.plot(rmse_learn)
plt.plot(rmse_test)
plt.title('RMSE on train/test data')
plt.xlabel('trees count')
plt.ylabel('rmse value')
plt.legend(['leanr', 'test']);

In [ ]:
#Get important features
cbr.feature_importances_


أهم الميزات هي "humidity_tr" و"atemp_tr" و"temp_tr"



### الجزء 9. إنشاء ميزات جديدة ووصف لهذه العملية


إضافة ميزات جديدة من "التاريخ والوقت"


In [ ]:
train_df['hour'] = train_df['datetime'].apply(lambda ts: ts.hour)
test_df['hour'] = test_df['datetime'].apply(lambda ts: ts.hour) 

train_df['weekday'] = train_df['datetime'].apply(lambda ts: ts.isoweekday())
test_df['weekday'] = test_df['datetime'].apply(lambda ts: ts.isoweekday())

train_df['month'] = train_df['datetime'].apply(lambda ts: ts.month)
test_df['month'] = test_df['datetime'].apply(lambda ts: ts.month) 

In [ ]:
draw_train_test_distribution('hour')
draw_train_test_distribution('weekday');
draw_train_test_distribution('month');


التوزيعات في مجموعات بيانات التدريب والاختبار متساوية.


In [ ]:
hours_mean_count = {}
hours_mean_casual = {}
hours_mean_registered = {}

hours_mean ={}
hours = np.unique(train_df['hour'])
print("hours:",hours)

temp_train_df =  train_df.join(y_train)

for h in hours:
    hours_mean_count[h] = temp_train_df.loc[temp_train_df['hour'] == h]['count'].mean()
    hours_mean_casual[h] = temp_train_df.loc[temp_train_df['hour'] == h]['casual'].mean()
    hours_mean_registered[h] = temp_train_df.loc[temp_train_df['hour'] == h]['registered'].mean()
    
    hours_mean[h] = [temp_train_df.loc[temp_train_df['hour'] == h]['count'].mean(),
                    temp_train_df.loc[temp_train_df['hour'] == h]['casual'].mean(),
                    temp_train_df.loc[temp_train_df['hour'] == h]['registered'].mean()]
    
hours_df = pd.DataFrame.from_dict(hours_mean, orient='index',
                        columns=['count', 'casual', 'registered'])  
hours_df['hours'] = hours

hours_df.plot(x='hours', y=['count', 'casual', 'registered'], kind='bar', 
              title = 'Measured bike use over 2 years',  legend = True );

del temp_train_df
gc.collect()


حسنًا. من المنطقي عمل الميزات بالساعة: الليل (23-0-6)، النهار (7-22)


In [ ]:
train_df['night'] = train_df['hour'].apply(lambda h: 1 if (h>=23) | (h<=6) else 0)
train_df['day'] = train_df['hour'].apply(lambda h: 1 if (h<23) & (h>6) else 0)

test_df['night'] = test_df['hour'].apply(lambda h: 1 if (h>=23) | (h<=6) else 0)
test_df['day'] = test_df['hour'].apply(lambda h: 1 if (h<23) & (h>6) else 0)


أنا أحب علم المثلثات)) [هنا](https://habr.com/company/ods/blog/325422/#data-i-vremya) و[هنا](https://medium.com/open-machine-learning-course/open-machine-learning-course-topic-6-feature-engineering-and-feature-selection-8b94f870706a) مثال رائع لكيفية استخدام علم المثلثات. سأقوم بتعديله قليلا.


In [ ]:
def make_harmonic_features(value, period=24):
    new_value = value * 2 * np.pi / period
    return np.cos(new_value), np.sin(new_value)

train_df['hour_cos'], train_df['hour_sin'] = make_harmonic_features(train_df['hour'])
test_df['hour_cos'], test_df['hour_sin'] = make_harmonic_features(test_df['hour'])


In [ ]:
train_df.head()


### الجزء العاشر. رسم منحنيات التدريب والتحقق من الصحة



بناء نموذج جديد مع معلمات جديدة


In [ ]:
X_train = train_df.drop(['holiday', 'datetime', 'temp', 'atemp', 'humidity'], axis=1)
X_test = test_df.drop(['holiday', 'datetime', 'temp', 'atemp', 'humidity'], axis=1)

In [ ]:
X_train.head()

In [ ]:
cat_features = [0, 1, 2, 7, 8, 9, 10, 11]

X_train_cbr, X_test_cbr, y_train_cbr, y_test_cbr = train_test_split(X_train, y_train, test_size=0.3, random_state=17, shuffle=True)

In [ ]:
%%time
obj_call_count = 0
cur_best_rmse = np.inf

trials = Trials()
best = fmin(fn=objective,
                     space=space,
                     algo=HYPEROPT_ALGO,
                     max_evals=N_HYPEROPT_PROBES,
                     trials=trials,
                     verbose=1)

In [ ]:
print('The best params:')
print(best)

In [ ]:
cbr = CatBoostRegressor(random_seed=17, 
                        eval_metric='RMSE', 
                        iterations=2000, 
                        max_depth=best['depth'], 
                        early_stopping_rounds=50, 
                        learning_rate=best['learning_rate'],
                        l2_leaf_reg=best['l2_leaf_reg'])
cbr.fit(X_train_cbr, y_train_cbr['count'], 
       eval_set=(X_test_cbr, y_test_cbr['count']), 
       cat_features=cat_features,
       silent=True,
       plot=True);

In [ ]:
rmse_learn = cbr.evals_result_['learn']['RMSE']
rmse_test = cbr.evals_result_['validation_0']['RMSE']

plt.plot(rmse_learn)
plt.plot(rmse_test)
plt.title('RMSE on train/test data')
plt.xlabel('trees count')
plt.ylabel('rmse value')
plt.legend(['leanr', 'test']);


### الجزء 11. التنبؤ بالعينات الاختبارية أو المحتجزة 


In [ ]:
%%time
cbr = CatBoostRegressor(random_seed=17, 
                        eval_metric='RMSE', 
                        iterations=2000, 
                        max_depth=best['depth'], 
#                         early_stopping_rounds=50, #because fit without eval_set
                        learning_rate=best['learning_rate'],
                        l2_leaf_reg=best['l2_leaf_reg'])
cbr.fit(X_train, y_train['count'], 
       cat_features=cat_features,
       silent=True,
       plot=False);

In [ ]:
y_pred = cbr.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test['count'], y_pred))
print('RMSE = ', rmse)


**يتطابق RMSE لعينات الإيقاف مع RMSE عند التحقق المتبادل!!!**



### الجزء 12. الاستنتاجات 
* ميزات جديدة حسنت النتيجة بشكل ملحوظ. يوضح لنا "cbr.feature_importances_" أهم المعلمات. إذا قمت بتحديد ميزات جديدة منها، يمكنك الحصول على نموذج أفضل.
* قد تؤدي بعض الميزات إلى تدهور النتيجة وسيكون من الضروري العثور عليها وإزالتها من النموذج.
* هنا يمكنك أيضًا تطبيق مجموعة من النماذج استنادًا إلى أهداف مختلفة ("غير رسمية" و"مسجلة" و"معدودة"). جرب أيضًا المكتبات الأخرى (lightgbm، XGBoost) وقارن النتائج. ويمكن أيضًا تطبيق مجموعة من النماذج بناءً على مكتبات مختلفة ))))
* يمكن استخدام الوظائف المنفذة **الهدف** و**get_catboost_params** لإعداد المعلمات الفائقة في مشاريع أخرى. أعتقد أن هذه بداية جيدة لإنشاء مكتبة أكثر مرونة يمكن استخدامها في مسابقات Kaggle
* لقد استخدمت وحدة المعالجة المركزية، ولكن يمكن استخدام وحدة معالجة الرسومات في catboost.